In [ ]:
import pandas as pd

In [ ]:
# Step 1: test load small chunk
df_demo = pd.read_csv("../data/raw/df_final_demo.csv")

# Step 2: load only part of big file
df_web_1 = pd.read_csv("../data/raw/df_final_web_data_pt_1.csv", nrows=1000)

In [ ]:
df_demo.head()

In [ ]:
# basic structure
df_demo.shape

In [ ]:
# column types + missing values
df_demo.info()

In [ ]:
# quick stats
df_demo.describe()

In [ ]:
# gender distribution
df_demo["gendr"].value_counts()

In [ ]:
# age overview
df_demo["clnt_age"].describe()

In [ ]:
df_exp = pd.read_csv("../data/raw/df_final_experiment_clients.csv")

In [ ]:
df_exp.head()

In [ ]:
df_exp["Variation"].value_counts()

In [ ]:
# missing values overview
df_demo.isna().sum()

In [ ]:
# Check rows with any missing values
df_demo[df_demo.isna().any(axis=1)]

In [ ]:
# remove rows with missing values
df_demo_clean = df_demo.dropna()

In [ ]:
df_demo_clean.shape

- Missing values were minimal (<0.05% of dataset), so rows were removed without significant data loss.

In [ ]:
df_demo_clean["gendr"].value_counts()

In [ ]:
df_demo_clean["gendr"].value_counts(normalize=True)

### Gender distribution

- The dataset contains three main categories: M, F, and U (unknown).
- A significant portion (~34%) of users have unspecified gender ("U").
- Removing this group would lead to substantial data loss, so it is kept as a separate category for analysis.

In [ ]:
df_demo_clean[["clnt_tenure_yr", "clnt_tenure_mnth"]].describe()

In [ ]:
# check relationship
df_demo_clean["clnt_tenure_yr"] * 12 - df_demo_clean["clnt_tenure_mnth"]

### Tenure variables consistency check

- The dataset contains both tenure in years and tenure in months.
- These variables are not perfectly consistent (years * 12 ≠ months).
- This suggests that tenure in months is more precise, while years may be rounded.
- Therefore, tenure in months is kept for further analysis, and tenure in years can be removed to avoid redundancy.

In [ ]:
df_demo_clean = df_demo_clean.drop(columns=["clnt_tenure_yr"])

In [ ]:
df_demo_clean.columns

In [ ]:
df_demo_clean["clnt_age"].describe()

In [ ]:
df_demo_clean["clnt_age"].value_counts().sort_index().head(10)

### Age distribution

- The average client age is around 46 years, indicating a middle-aged user base.
- The dataset includes a wide age range (approx. 13 to 96).
- Some age values appear as decimals (e.g., 13.5, 14.5), which may indicate imprecise or transformed data.
- Despite this, age is kept as a continuous variable for analysis.

In [ ]:
df_demo_clean["clnt_age"].hist(bins=30)

### Age distribution insights

- The majority of clients fall between 30 and 60 years old.
- The distribution is slightly right-skewed, with fewer older clients extending the tail.
- Very few clients are younger than 20.
- This suggests that the primary users of the platform are middle-aged rather than young clients.

In [ ]:
df_demo_clean[["clnt_age", "logons_6_mnth", "calls_6_mnth"]].corr()

### Client behavior insights

- Age shows almost no correlation with user activity (logons: ~0.08, calls: ~0.03).
- This suggests that client behavior is not strongly influenced by age.
- In contrast, logons and calls are highly correlated (~0.82).
- This indicates that highly engaged users tend to interact frequently across multiple channels.

In [ ]:
df_demo_clean.groupby("gendr")[["logons_6_mnth", "calls_6_mnth"]].mean()

### Behavior differences by gender

- There are small differences in activity levels across genders.
- Male clients show slightly higher average logons and calls compared to female clients.
- However, the differences are relatively minor overall.
- This suggests that user behavior is fairly consistent across gender groups.
- The "X" category has very few observations and is not statistically reliable.

In [ ]:
df_demo_clean["logons_6_mnth"].describe()

### Engagement level insights

- The average number of logons over 6 months is around 5.6, with a median of 5.
- Most users fall between 4 and 7 logons, indicating moderate engagement.
- The distribution is relatively tight, with few extreme values.
- This suggests that user activity levels are fairly consistent across the client base.

### Key insights from demographic analysis

- The majority of clients are middle-aged (30–60 years old).
- Gender distribution is relatively balanced across the dataset.
- Age shows almost no correlation with user activity.
- Behavior is also fairly consistent across gender groups, with only minor differences.

### Behavior insights

- Logons and calls are strongly correlated (~0.82), indicating consistent engagement patterns.
- Users who are active in one channel tend to be active in others as well.
- Overall engagement levels are moderate, with most users logging in 4–7 times over 6 months.
- There are no clear high-activity or low-activity segments based on demographics.

### Implications for next steps

- Demographic variables (age, gender) may not be strong predictors of behavior.
- Behavioral metrics should be the focus for further analysis.
- We should confirm whether demographic patterns differ after merging with experiment data.

### Data cleaning decisions (demographics)

- Rows with missing values were removed due to low impact.
- Tenure in months was kept as the main variable, as it is more precise than years.
- Tenure in years was dropped due to inconsistency.
- Gender values were kept as is, including "U" (unknown).

In [ ]:
df_demo_clean.to_csv("../data/clean/df_demo_clean.csv", index=False)

In [ ]:
df_demo_clean["client_id"].duplicated().sum()

- No duplicate client_ids found → dataset is unique at client level and ready for merging.

In [ ]:
df_demo["bal"].describe()
df_demo[df_demo["bal"] < 0]

- Balance values are all non-negative and within a reasonable range → no cleaning required.

In [31]:
df_demo_clean["clnt_tenure_mnth"].describe()

count    70594.000000
mean       150.659999
std         82.090264
min         33.000000
25%         82.000000
50%        136.000000
75%        192.000000
max        749.000000
Name: clnt_tenure_mnth, dtype: float64

In [35]:
df_demo_clean["tenure_group"] = pd.cut(
    df_demo_clean["clnt_tenure_mnth"],
    bins=[0, 60, 120, 240, 1000],
    labels=["<5y", "5–10y", "10–20y", "20y+"]
)

In [36]:
df_demo_clean["tenure_group"].value_counts(normalize=True)

tenure_group
10–20y    0.388957
5–10y     0.368388
20y+      0.159461
<5y       0.083194
Name: proportion, dtype: float64

The client base is heavily skewed towards long-standing users, with over 75% having more than 5 years of tenure.
This suggests that the platform relies primarily on established customers rather than newly acquired users.

In [37]:
df_demo_clean.groupby("tenure_group")[["logons_6_mnth", "calls_6_mnth"]].mean()

,logons_6_mnth,calls_6_mnth
tenure_group,,
<5y,5.514218,3.309552
5–10y,5.552103,3.377451
10–20y,5.451672,3.265169
20y+,5.908501,3.718220


While the client base is predominantly long-standing, tenure does not appear to strongly influence engagement levels, which remain fairly stable across all groups.

## Key Insights from Demographic Analysis

- The typical client is middle-aged and long-standing (5–20 years tenure), indicating a mature and stable customer base.

- Over 75% of clients have more than 5 years of tenure, with very few new users, suggesting the platform relies primarily on established customers rather than recent growth.

- User engagement is moderate and consistent, with most clients logging in 4–7 times over a 6-month period.

- Demographic factors (age, gender, tenure) do not strongly influence user behavior, as engagement levels remain relatively stable across all groups.

- Logons and calls are strongly correlated, indicating that highly engaged users tend to interact frequently across multiple channels.

- This suggests that improvements should focus on user experience and behavioral drivers rather than demographic segmentation.